# 13 (DS) — Sampling, Splits & Segments

**Data Scientist perspective.** Reproducible sampling, train/test splitting, stratified draws, cohort pivots and decile analysis — the mechanics behind fair evaluation and segmentation.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Population

600 customers across four regions.

In [ ]:
import random
import pandas as pd

random.seed(11)
REGIOES = ["SP", "RJ", "MG", "RS"]
rows = [{
    "cliente_id": i,
    "regiao": random.choices(REGIOES, weights=[5, 3, 2, 1])[0],
    "idade": random.randint(18, 70),
    "ltv": round(random.expovariate(1 / 800), 2),
} for i in range(1, 601)]
clientes = session.createDataFrame(pd.DataFrame(rows))
clientes.groupBy("regiao").count().orderBy("regiao").show()

## 2. Reproducible sampling

Same seed → same sample. Different seed → different sample.

In [ ]:
s1 = clientes.sample(fraction=0.2, seed=42)
s2 = clientes.sample(fraction=0.2, seed=42)
s3 = clientes.sample(fraction=0.2, seed=99)
print("seed 42 twice:", s1.count(), s2.count(), "-> equal:", s1.count() == s2.count())
print("seed 99:", s3.count())

## 3. Train/test split

`randomSplit` returns partitioned DataFrames.

In [ ]:
train, test = clientes.randomSplit([0.8, 0.2], seed=7)
print("train:", train.count(), "| test:", test.count())
print("overlap-free union:", train.union(test).distinct().count() == clientes.count())

## 4. Stratified sampling — `sampleBy`

Keep every region represented regardless of its share.

In [ ]:
fractions = {r: 0.3 for r in REGIOES}
sample = clientes.stat.sampleBy("regiao", fractions, sampleByColumns=["regiao"])
print(type(sample).__name__, "| rows:", len(sample))
print(sample["regiao"].value_counts().to_dict())

## 5. Segments by LTV deciles

`ntile(10)` over an ordered window buckets customers into value deciles.

In [ ]:
from irispark import Window
from irispark.functions import ntile, avg, count, col

w = Window.orderBy(col("ltv").desc())
deciles = clientes.withColumn("decil", ntile(10).over(w))
deciles.groupBy("decil").agg(
    count("cliente_id").alias("n"),
    avg("ltv").alias("ltv_medio"),
    avg("idade").alias("idade_media"),
).orderBy("decil").show()

## 6. Cohort pivot — region × age band

Conditional buckets + pivot produce a management-style view.

In [ ]:
from irispark.functions import when, lit

banded = clientes.withColumn(
    "faixa_idade",
    when(col("idade") < 30, lit("18-29"))
    .when(col("idade") < 45, lit("30-44"))
    .when(col("idade") < 60, lit("45-59"))
    .otherwise(lit("60+")),
)
banded.groupBy("regiao").pivot("faixa_idade").count().orderBy("regiao").show()

## 7. Materialize the segment table

Segments are reusable — persist them for the marketing team.

In [ ]:
banded.select("cliente_id", "regiao", "faixa_idade", "ltv") \
    .write.mode("overwrite").saveAsTable("cliente_segmentos_demo")
print("materialized cliente_segmentos_demo:", session.table("cliente_segmentos_demo").count(), "rows")

## 8. Cleanup

In [ ]:
session.sql("DROP TABLE IF EXISTS cliente_segmentos_demo")
print("dropped cliente_segmentos_demo")

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")